# Choosing Drivers of Unit Sales
## Which levers should the retailer pull?

**The business question.** A retailer sells one product across 50 shops. Shelf space, promotion spend, price, footfall, the competitor's price and the weather all vary from shop to shop. Which of them really move unit sales, by how much, and which are a waste of the manager's attention?

Work through five steps: load and look, run the model, read it in plain English, check it against JMP, then say it to a manager. **Python leads and JMP is the check.**

### How to use this notebook

- Run cells in order with **Shift+Enter**. The data file is `../data/retail_sales_data.csv`. Prices and spend are in pounds, shelf space in centimetres and temperature in degrees Celsius.
- Copilot is on. Where a cell says *Ask Copilot*, write a short prompt, run what it gives you, and check against the numbers shown.

---

## Part 1: Exercise: what drives unit sales

### Step 1: Load and look

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan

shops = pd.read_csv("../data/retail_sales_data.csv")
print(shops.shape, "| missing values:", int(shops.isna().sum().sum()))
shops.describe().round(1)

In [ ]:
def fit_ols(data, y, drivers):
    # Fit y on a list of drivers. The constant is the intercept.
    return sm.OLS(data[y], sm.add_constant(data[drivers])).fit()

def vifs(data, drivers):
    # Variance inflation factor for each driver: how much the others overlap with it
    X = sm.add_constant(data[drivers])
    return pd.Series([variance_inflation_factor(X.values, i + 1) for i in range(len(drivers))], index=drivers).round(2)

levers = ["Product_Price_GBP", "Shelf_Space_cm", "Promotion_Spend_GBP", "Footfall_Score", "Competitor_Price_GBP", "Temperature_C"]
shops[levers + ["Units_Sold"]].corr()["Units_Sold"].round(3)

### Step 2: Run the model

Fit units sold on all six drivers.

In [ ]:
full = fit_ols(shops, "Units_Sold", levers)

print("R-squared", round(full.rsquared, 3), "| adjusted", round(full.rsquared_adj, 3), "| typical miss", round(np.sqrt(full.mse_resid), 1), "units")
pd.DataFrame({"Slope": full.params, "Lower 95%": full.conf_int()[0], "Upper 95%": full.conf_int()[1],
              "p-value": full.pvalues, "VIF": [np.nan] + list(vifs(shops, levers))}).round(3)

### Step 3: Read it in plain English

Each slope is the change in units sold for one extra unit of that driver, **holding the other five constant.**

- **Shelf space:** about 2.8 extra units per extra centimetre.
- **Promotion spend:** about 0.81 extra units per extra pound.
- **Price:** about 12 fewer units per extra pound.
- **Footfall, competitor price and temperature:** their intervals include zero. The data gives no reason to think they move sales in this file.

All the VIFs are close to 1, so none of the drivers overlaps with the others. That makes the reading clean.

Now the surprise. Price on its own:

In [ ]:
price_alone = fit_ols(shops, "Units_Sold", ["Product_Price_GBP"])
print(f"Price alone:        slope {price_alone.params['Product_Price_GBP']:7.2f}   p = {price_alone.pvalues['Product_Price_GBP']:.3f}   R-squared {price_alone.rsquared:.3f}")
print(f"Price in the model: slope {full.params['Product_Price_GBP']:7.2f}   p = {full.pvalues['Product_Price_GBP']:.3f}")

The price slope is about the same size in both, but on its own it looks like noise (p about 0.28) and in the full model it is significant (p about 0.04). Shelf space and promotion explain a big share of the variation between shops. Once they are in the model, the leftover variation is small enough for the price effect to show. Adding a driver can *sharpen* another driver, not only change it.

### Step 4: Build the simpler model

Drop the weakest driver by p-value, one at a time, until everything left is significant. Change the list below to try a different set.

In [ ]:
keep = levers.copy()
while True:
    m = fit_ols(shops, "Units_Sold", keep)
    p = m.pvalues.drop("const")
    if p.max() <= 0.05:
        break
    print(f"Drop {p.idxmax():<22} p = {p.max():.3f}")
    keep.remove(p.idxmax())

reduced = m
print("\nLeft with:", keep)
print("R-squared", round(reduced.rsquared, 3), "| adjusted", round(reduced.rsquared_adj, 3))
pd.DataFrame({"Slope": reduced.params, "Lower 95%": reduced.conf_int()[0], "Upper 95%": reduced.conf_int()[1], "p-value": reduced.pvalues}).round(3)

Adjusted R-squared goes *up* when the three useless drivers go, from about 0.797 to 0.807. R-squared barely moves. That is the sign that nothing useful was lost.

Check the model's behaviour before trusting it.

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 3.6))
ax.scatter(reduced.fittedvalues, reduced.resid, s=14)
ax.axhline(0, color="grey")
ax.set_xlabel("Fitted units sold"); ax.set_ylabel("Residual"); ax.set_title("Residuals against fitted")
plt.show()

print("Breusch-Pagan p-value (large = spread of misses looks even):", round(het_breuschpagan(reduced.resid, reduced.model.exog)[1], 2))
infl = reduced.get_influence()
print("Largest Cook's distance:", round(infl.cooks_distance[0].max(), 2))

### Step 5: Check against JMP

In JMP, **Analyze > Fit Model** with Units Sold as Y and all six drivers as model effects. You should see an R-squared of about **0.822** and adjusted about **0.797**. Slopes about **-11.97** for price, **+2.85** for shelf space, **+0.81** for promotion spend, with footfall, competitor price and temperature not significant. Then remove the three non-significant drivers and refit: R-squared about **0.819**, adjusted about **0.807**. Compare each number with your Python output.

### Try this

Change what you ask the model. In the cell below, replace the driver list with a different set, for example only the three levers the retailer controls (`Product_Price_GBP`, `Shelf_Space_cm` and `Promotion_Spend_GBP`), or all but price. See which results survive.

In [ ]:
try_levers = ["Shelf_Space_cm", "Promotion_Spend_GBP"]   # change this list, then re-run

m = fit_ols(shops, "Units_Sold", try_levers)
print(m.params.round(3).to_dict())
print("Adjusted R-squared:", round(m.rsquared_adj, 3), "| p-values:", m.pvalues.round(4).to_dict())

### A costed option

Managers do not buy slopes, they buy returns. One extra pound of promotion goes with about 0.81 extra units, so each unit must earn more than £1.23 in margin (1 / 0.81) for the pound to pay for itself. Fill in the margin per unit that you would assume, and work out where promotion pays.

In [ ]:
promo_slope = reduced.params["Promotion_Spend_GBP"]
low, high = reduced.conf_int().loc["Promotion_Spend_GBP"]

print("Units per extra pound of promotion:", round(promo_slope, 2), f"(95% interval {low:.2f} to {high:.2f})")
print("Break-even margin per unit (pounds):", round(1 / promo_slope, 2), f"(from {1/high:.2f} to {1/low:.2f})")

margin_per_unit = 2.00   # change this to the margin you would assume, in pounds
print(f"At a margin of £{margin_per_unit:.2f} a unit, £1 of promotion returns about £{promo_slope * margin_per_unit:.2f}")

Break-even is about £1.23 of margin per unit. If a unit earns more than that, the data supports more promotion *in shops like these*; if it earns less, it does not. That is a decision rule a manager can use. It rests on 50 shops from one retailer, so it does not promise that the same effect will repeat in other shops.

### Commentary cell

1. Which levers matter, and which can the manager stop worrying about?
2. Price on its own looked like noise. Why did it matter in the full model?
3. Which would you launch, change or stop: a bigger promotion budget, more shelf space, or a price cut? What would it cost to be wrong?

*Your commentary here*

### Say it to a manager

Two sentences, in business language. Name the levers that matter, give one effect with its units, and say one thing the data cannot tell the manager.

*Your two sentences here*

### Manager note

Write a note of **150 words or fewer** to the retail director. It should say which levers to pull, give the size of the effect with its units, give one costed option, and say what the data cannot promise. Do not claim that the same effects would hold in other stores. Save it in the cell below and commit the notebook to your repository.

*Your note here*

### Stretch

A blank space for your own idea. For example: ask Copilot for a chart of the slopes with their 95% intervals, or find the three shops whose sales the model predicts worst and say what they have in common.

In [ ]:
# Stretch